In [1]:
pip install razdel

In [2]:
pip install pymorphy3

In [3]:
pip install WordCloud

In [4]:
pip install datasets

In [5]:
pip install graphviz

#Библиотеки

In [6]:
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import warnings
from collections import Counter
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score


import nltk
from nltk.corpus import stopwords
from razdel import tokenize as razdel_tokenize
import pymorphy3
from wordcloud import WordCloud


import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, Trainer, TrainingArguments, AutoModelForSequenceClassification, AutoModel

from datasets import Dataset as HFDataset

from graphviz import Digraph

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

if torch.backends.mps.is_available():
    device = torch.device("mps")
    torch.mps.manual_seed(RANDOM_STATE)
elif torch.cuda.is_available():
    device = torch.device("cuda")
    torch.cuda.manual_seed_all(RANDOM_STATE)
else:
    device = torch.device("cpu")

print(f"✅ Используемое устройство: {device}")


✅ Используемое устройство: cuda


#Первоначальная обработка данных

In [7]:
def norm_text(x):
  if pd.isna(x):
    return "<EMPTY>"
  x = str(x)
  x = x.replace("\n", " ").replace("\t", " ")
  x = " ".join(x.split())
  return x

def build_text(row):
  return (
      f"<TITLE> {row['title']}\n"
      f"<EXP> {int(row['experience_from'])}\n"
      f"<LOC> {row['location']}\n"
      f"<COMPANY> {row['company']}\n"
      f"<SKILLS> {row['skills']}\n"
      f"<DESC> {row['description']}"
      )

def prepare_data(df: pd.DataFrame) -> pd.DataFrame:
    prepared_df = df.copy()

    prepared_df['title']       = prepared_df['title'].apply(norm_text)
    prepared_df['location']    = prepared_df['location'].apply(norm_text)
    prepared_df['company']     = prepared_df['company'].apply(norm_text)
    prepared_df['skills']      = prepared_df['skills'].fillna('').apply(norm_text)
    prepared_df['description'] = prepared_df['description'].fillna('').apply(norm_text)

    prepared_df["full_text"] = prepared_df.apply(build_text, axis=1)

    prepared_df["full_text"] = (
        prepared_df["full_text"]
        .str.replace(r'<[^>]+>', ' ', regex=True)
        .str.replace(r'\n\n+', '\n', regex=True)
        .str.replace(r'\t+', ' ', regex=True)
        .str.replace(r' +', ' ', regex=True)
        .str.strip()
    )

    return prepared_df

In [8]:
full_train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

In [9]:
full_train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16667 entries, 0 to 16666
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   title            16667 non-null  object 
 1   location         16667 non-null  object 
 2   company          16667 non-null  object 
 3   skills           10842 non-null  object 
 4   description      16667 non-null  object 
 5   experience_from  16667 non-null  float64
 6   salary_from      16667 non-null  float64
 7   log_salary_from  16667 non-null  float64
dtypes: float64(3), object(5)
memory usage: 1.0+ MB


In [10]:
full_train_df.isna().sum()

,0
title,0
location,0
company,0
skills,5825
description,0
experience_from,0
salary_from,0
log_salary_from,0


In [11]:
test_df.isna().sum()

,0
title,0
location,0
company,0
skills,2014
description,0
experience_from,0


In [12]:
test_df.head(5)

,title,location,company,skills,description,experience_from
0,Ведущий программист 1С (г. Санкт-Петербург),Санкт-Петербург,Коннект персонал,"1С программирование, MS SQL Server, 1C: ERP, О...",Крупнейший производственный комплекс легкой пр...,3.0
1,Ресерчер (поиск товаров на маркетплейсах),Москва,Right Choice,"Конкурентная аналитика, Аналитические исследов...","Мы молодая команда селлеров, состоящая из 12 ч...",1.0
2,Системный администратор,Нижний Новгород,Меридиан,"Администрирование сетевого оборудования, Админ...",О компании: Уже более 30 лет мы успешно прои...,1.0
3,Инженер по интеграции систем защиты информации,Новосибирск,СофтМолл,"Информационная безопасность, Аналитическое мыш...","SoftMall – это аккредитованная IT-компания, к...",1.0
4,Ведущий менеджер по работе с маркетплейсом Wil...,Москва,ДЖЕЙКЕТ РАБОТА,NaN,Вакансия компании: Brosco Компания Brosco зан...,1.0


In [13]:
full_train_df = prepare_data(full_train_df)

In [14]:
test_df = prepare_data(test_df)

In [15]:
train_df, val_df = train_test_split(
    full_train_df,
    test_size=0.2,
    random_state=42
)

In [16]:
y_train = train_df['log_salary_from']
y_val = val_df['log_salary_from']

In [17]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
save_path = "/content/drive/MyDrive/nlp_reg_data/train_df.csv"
train_df.to_csv(save_path, index=False, encoding='utf-8')

In [19]:
train_df.to_csv("train_df.csv")

In [20]:
val_df.to_csv("val_df.csv")

In [21]:
test_df.to_csv("test_df.csv")

In [22]:
save_path = "/content/drive/MyDrive/nlp_reg_data/val_df.csv"
val_df.to_csv(save_path, index=False, encoding='utf-8')

In [23]:
save_path = "/content/drive/MyDrive/nlp_reg_data/test_df.csv"
test_df.to_csv(save_path, index=False, encoding='utf-8')

# Обучение Берта+линейная часть

In [24]:
class BertRegressor(nn.Module):
    def __init__(self, bert_name):
        super(BertRegressor, self).__init__()
        self.bert = AutoModel.from_pretrained(
            bert_name,
            hidden_dropout_prob=0.1,
            attention_probs_dropout_prob=0.1
        )
        self.regressor = nn.Linear(self.bert.config.hidden_size, 1)
        self.loss_fn = nn.HuberLoss()

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output
        logits = self.regressor(pooled_output).squeeze(-1)

        if labels is not None:
            loss = self.loss_fn(logits, labels)
            return {"loss": loss, "logits": logits}
        return {"logits": logits}

In [25]:
MODEL_NAME = 'ai-forever/ruBert-base'
tokenizer   = AutoTokenizer.from_pretrained(MODEL_NAME)

In [26]:
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

In [27]:
train_bert_df = train_df.copy()
val_bert_df = val_df.copy()

In [28]:
train_bert_df = train_bert_df[['full_text', 'log_salary_from']].rename(columns={'full_text': 'text', 'log_salary_from': 'label'})
val_bert_df = val_bert_df[['full_text', 'log_salary_from']].rename(columns={'full_text': 'text', 'log_salary_from': 'label'})

In [29]:
train_hf_dataset = HFDataset.from_pandas(train_bert_df)
val_hf_dataset = HFDataset.from_pandas(val_bert_df)

In [30]:
tokenized_train = train_hf_dataset.map(tokenize_function, batched=True, remove_columns=['text'])
tokenized_val = val_hf_dataset.map(tokenize_function, batched=True, remove_columns=['text'])

Map:   0%|          | 0/13333 [00:00<?, ? examples/s]

Map:   0%|          | 0/3334 [00:00<?, ? examples/s]

In [31]:
model = BertRegressor(MODEL_NAME).to(device)

In [32]:
training_args = TrainingArguments(
    output_dir="/content/Model_bert",
            eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=200,
        learning_rate= 3.5e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        save_total_limit=2,
        num_train_epochs=4,
        weight_decay= 0.09,
        warmup_steps= 1500,
        lr_scheduler_type="polynomial",
        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported(),
        seed=42,
        report_to="tensorboard",
        load_best_model_at_end=True,
        metric_for_best_model="r2",
        greater_is_better=True,
    )

In [33]:
def compute_r2_score(y_true, y_pred):
    score = r2_score(y_true, y_pred)
    print(f"R2 Score: {score:.6f}")
    return score

In [34]:
def compute_metrics_for_trainer(eval_pred):
    predictions, labels = eval_pred
    predictions = predictions.flatten()
    labels = labels.flatten()
    r2 = compute_r2_score(labels, predictions)
    return {"r2": r2}

In [35]:
tokenized_train = tokenized_train.remove_columns(["token_type_ids"])
tokenized_val   = tokenized_val.remove_columns(["token_type_ids"])

In [36]:
trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        compute_metrics=compute_metrics_for_trainer,
        processing_class=tokenizer
    )

In [37]:
trainer.train()


Epoch,Training Loss,Validation Loss,R2
1,0.068000,0.081449,0.570827
2,0.058600,0.061363,0.676920
3,0.038400,0.053777,0.716555
4,0.019500,0.045181,0.761905


R2 Score: 0.570827
R2 Score: 0.676920
R2 Score: 0.716555
R2 Score: 0.761905


TrainOutput(global_step=3336, training_loss=0.1473216729627239, metrics={'train_runtime': 7693.42, 'train_samples_per_second': 6.932, 'train_steps_per_second': 0.434, 'total_flos': 0.0, 'train_loss': 0.1473216729627239, 'epoch': 4.0})

# Сохранение модели

In [38]:
FINAL_DIR = "/content/Model_bert_final"
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print("Saved to:", FINAL_DIR)

Saved to: /content/Model_bert_final


In [39]:
from google.colab import drive
drive.mount("/content/drive")

!cp -r /content/Model_bert_final /content/drive/MyDrive/trainer_output

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Загрузка готовой модели и eval

In [ ]:
class BertRegressor(nn.Module):
    def __init__(self, bert_name):
        super(BertRegressor, self).__init__()
        self.bert = AutoModel.from_pretrained(
            bert_name,
            hidden_dropout_prob=0.1,
            attention_probs_dropout_prob=0.1
        )
        self.regressor = nn.Linear(self.bert.config.hidden_size, 1)
        self.loss_fn = nn.HuberLoss()

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output
        logits = self.regressor(pooled_output).squeeze(-1)

        if labels is not None:
            loss = self.loss_fn(logits, labels)
            return {"loss": loss, "logits": logits}
        return {"logits": logits}

model_dir = "/content/drive/MyDrive/Model_bert_final"

In [ ]:
from safetensors.torch import load_file

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_dir)
state_dict = load_file(f"{model_dir}/model.safetensors")

In [ ]:
model = BertRegressor("ai-forever/ruBert-base")
model.load_state_dict(state_dict)
model.to(device)
model.eval()

BertRegressor(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(120138, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwis

In [ ]:
texts_to_predict = train_df["full_text"].tolist()

In [ ]:
predictions_train = []
for i in range(0, len(texts_to_predict), 32):
  if i % 3000 == 0:
    print('3000')
  batch_texts = texts_to_predict[i:i + 32]
  inputs = tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True,
                           max_length=512)
  inputs = {k: v.to("cuda") for k, v in inputs.items()}
  inputs.pop("token_type_ids", None)
  with torch.no_grad():
    outputs = model(**inputs)
    preds = outputs["logits"].cpu().numpy().flatten()
    predictions_train.extend(preds)


3000
3000


In [ ]:
texts_to_predict = val_df["full_text"].tolist()

In [ ]:
len(texts_to_predict)

3334

In [ ]:
predictions_val = []
for i in range(0, len(texts_to_predict), 32):
  if i % 3000 == 0:
    print('3000')
  batch_texts = texts_to_predict[i:i + 32]
  inputs = tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True,
                           max_length=512)
  inputs = {k: v.to("cuda") for k, v in inputs.items()}
  inputs.pop("token_type_ids", None)
  with torch.no_grad():
    outputs = model(**inputs)
    preds = outputs["logits"].cpu().numpy().flatten()
    predictions_val.extend(preds)


3000


In [ ]:
len(predictions_val)

3334

In [ ]:
texts_to_predict = test_df["full_text"].tolist()

In [ ]:
len(texts_to_predict)

5556

In [ ]:
predictions_test = []
for i in range(0, len(texts_to_predict), 32):
  if i % 3000 == 0:
    print('3000')
  batch_texts = texts_to_predict[i:i + 32]
  inputs = tokenizer(batch_texts, return_tensors="pt", padding=True, truncation=True,
                           max_length=512)
  inputs = {k: v.to("cuda") for k, v in inputs.items()}
  inputs.pop("token_type_ids", None)
  with torch.no_grad():
    outputs = model(**inputs)
    preds = outputs["logits"].cpu().numpy().flatten()
    predictions_test.extend(preds)


3000


In [ ]:
len(predictions_test)

5556

In [ ]:
import numpy as np
from google.colab import files

np.save("/content/predictions_train.npy", np.array(predictions_train))

In [ ]:
import numpy as np
from google.colab import files

np.save("/content/predictions_val.npy", np.array(predictions_val))

In [ ]:
import numpy as np
from google.colab import files

np.save("/content/predictions_test.npy", np.array(predictions_test))